In [ ]:
import numpy as np
import pandas as pd
import sklearn
import torch
import scipy.stats

from tqdm import tqdm

from scipy.special import expit
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

In [ ]:
# PARAMETERS

NUM_RESAMPLES = 300
NUM_TRIALS = 100

SEED = 120013

FONT_SIZE = 20

SAVE = False
LOAD = True

rng = np.random.default_rng(seed=SEED)

# Breast Cancer

### Generate semi-synthetic data

In [ ]:
df_bc = pd.read_csv('./data/breastcancer_data.csv')
df_bc

In [ ]:
X_bc = df_bc.iloc[:,1:].to_numpy()
X_bc = np.concatenate([np.ones(shape=(X_bc.shape[0],1)),X_bc],axis=1)
y_bc = df_bc.iloc[:,0].to_numpy()

In [ ]:
if LOAD:
    w_true_bc = np.load('saved_data/w_true_bc.npy')
else:
    LR = LogisticRegression()
    LR.fit(X_bc[:,1:], y_bc)
    w_true_bc = np.ones(shape=(X_bc.shape[1],1))
    w_true_bc[0] = LR.intercept_
    w_true_bc[1:] = LR.coef_.reshape(-1, 1)

In [ ]:
logit_true_bc = X_bc @ w_true_bc
probs_true_bc = expit(logit_true_bc)
om_probs_true_bc = expit(-logit_true_bc)

### Initial logreg on semi-synthetic data

In [ ]:
if LOAD:
    logit_pred_bc = np.load('saved_data/logit_pred_bc.npy')
    y_obs_bc      = np.load('saved_data/y_obs_bc.npy')

else:
    logit_pred_bc = np.zeros((X_bc.shape[0], NUM_TRIALS))
    y_obs_bc = np.zeros((X_bc.shape[0], NUM_TRIALS))
    
    for i in range(NUM_TRIALS):
        LR = LogisticRegression(C=np.inf, fit_intercept=False, solver='lbfgs')
        y_obs_bc[:,i] = rng.binomial(1, probs_true_bc[:,0])
        LR.fit(X_bc, y_obs_bc[:,i])
        w = LR.coef_.reshape(-1, 1)
    
        logit_pred_bc[:,i] = (X_bc @ w).reshape(-1)

In [ ]:
probs_pred_bc = expit(logit_pred_bc)
om_probs_pred_bc = expit(-logit_pred_bc)

### Compute regret

In [ ]:
if LOAD:
    resamp_from_true_logodds_bc = np.load('saved_data/resamp_from_true_logodds_bc.npy')
    resamp_from_pred_logodds_bc = np.load('saved_data/resamp_from_pred_logodds_bc.npy')

else:
    resamp_from_true_logodds_bc = np.zeros((X_bc.shape[0], NUM_TRIALS, NUM_RESAMPLES))
    resamp_from_pred_logodds_bc = np.zeros((X_bc.shape[0], NUM_TRIALS, NUM_RESAMPLES))
    
    for i in tqdm(range(NUM_TRIALS)):
        
        for j in range(NUM_RESAMPLES):
            
            LR_from_true = LogisticRegression(C=np.inf, fit_intercept=False, solver='lbfgs')
            LR_from_true.fit(X_bc, rng.binomial(1, probs_true_bc[:,0]))
            w_from_true = LR_from_true.coef_.reshape(-1, 1)
                
            LR_from_pred = LogisticRegression(C=np.inf, fit_intercept=False, solver='lbfgs')
            LR_from_pred.fit(X_bc, rng.binomial(1, probs_pred_bc[:,i]))
            w_from_pred = LR_from_pred.coef_.reshape(-1, 1)
    
            resamp_from_true_logodds_bc[:,i,j] = (X_bc @ w_from_true).reshape(-1)
            resamp_from_pred_logodds_bc[:,i,j] = (X_bc @ w_from_pred).reshape(-1)

In [ ]:
regret_logit_true_bc = np.var(resamp_from_true_logodds_bc, axis=2, ddof=1)
regret_logit_pred_bc = np.var(resamp_from_pred_logodds_bc, axis=2, ddof=1)

regret_probs_true_bc = np.var(expit(resamp_from_true_logodds_bc), axis=2, ddof=1)
regret_probs_pred_bc = np.var(expit(resamp_from_pred_logodds_bc), axis=2, ddof=1)

# Bank

### Generate semi-synthetic data

In [ ]:
df_bk = pd.read_csv('./data/bank_data.csv')
df_bk

In [ ]:
X_bk = df_bk.iloc[:,1:].to_numpy()
X_bk = np.concatenate([np.ones(shape=(X_bk.shape[0],1)),X_bk],axis=1)
y_bk = df_bk.iloc[:,0].to_numpy()

In [ ]:
_, R = np.linalg.qr(X_bk)
X_bk = X_bk[:,np.abs(np.diag(R)) > 1e-10]

In [ ]:
if LOAD:
    w_true_bk = np.load('saved_data/w_true_bk.npy')
else:
    LR = LogisticRegression()
    LR.fit(X_bk[:,1:], y_bk)
    w_true_bk = np.ones(shape=(X_bk.shape[1],1))
    w_true_bk[0] = LR.intercept_
    w_true_bk[1:] = LR.coef_.reshape(-1, 1)

In [ ]:
logit_true_bk = X_bk @ w_true_bk
probs_true_bk = expit(logit_true_bk)
om_probs_true_bk = expit(-logit_true_bk)

### Initial logreg on semi-synthetic data

In [ ]:
if LOAD:
    logit_pred_bk = np.load('saved_data/logit_pred_bk.npy')
    y_obs_bk      = np.load('saved_data/y_obs_bk.npy')

else:
    logit_pred_bk = np.zeros((X_bk.shape[0], NUM_TRIALS))
    y_obs_bk = np.zeros((X_bk.shape[0], NUM_TRIALS))
    
    for i in range(NUM_TRIALS):
        LR = LogisticRegression(C=np.inf, fit_intercept=False, solver='lbfgs')
        y_obs_bk[:,i] = rng.binomial(1, probs_true_bk[:,0])
        LR.fit(X_bk, y_obs_bk[:,i])
        w = LR.coef_.reshape(-1, 1)
    
        logit_pred_bk[:,i] = (X_bk @ w).reshape(-1)

In [ ]:
probs_pred_bk = expit(logit_pred_bk)
om_probs_pred_bk = expit(-logit_pred_bk)

### Compute regret

In [ ]:
if LOAD:
    resamp_from_true_logodds_bk = np.load('saved_data/resamp_from_true_logodds_bk.npy')
    resamp_from_pred_logodds_bk = np.load('saved_data/resamp_from_pred_logodds_bk.npy')

else:
    resamp_from_true_logodds_bk = np.zeros((X_bk.shape[0], NUM_TRIALS, NUM_RESAMPLES))
    resamp_from_pred_logodds_bk = np.zeros((X_bk.shape[0], NUM_TRIALS, NUM_RESAMPLES))
    
    for i in tqdm(range(NUM_TRIALS)):
        
        for j in range(NUM_RESAMPLES):
            
            LR_from_true = LogisticRegression(C=np.inf, fit_intercept=False, solver='lbfgs')
            LR_from_true.fit(X_bk, rng.binomial(1, probs_true_bk[:,0]))
            w_from_true = LR_from_true.coef_.reshape(-1, 1)
                
            LR_from_pred = LogisticRegression(C=np.inf, fit_intercept=False, solver='lbfgs')
            LR_from_pred.fit(X_bk, rng.binomial(1, probs_pred_bk[:,i]))
            w_from_pred = LR_from_pred.coef_.reshape(-1, 1)
    
            resamp_from_true_logodds_bk[:,i,j] = (X_bk @ w_from_true).reshape(-1)
            resamp_from_pred_logodds_bk[:,i,j] = (X_bk @ w_from_pred).reshape(-1)

In [ ]:
regret_logit_true_bk = np.var(resamp_from_true_logodds_bk, axis=2, ddof=1)
regret_logit_pred_bk = np.var(resamp_from_pred_logodds_bk, axis=2, ddof=1)

regret_probs_true_bk = np.var(expit(resamp_from_true_logodds_bk), axis=2, ddof=1)
regret_probs_pred_bk = np.var(expit(resamp_from_pred_logodds_bk), axis=2, ddof=1)

# Apnea

### Generate semi-synthetic data

In [ ]:
df_ap = pd.read_csv('./data/apnea/apnea_data.csv')
df_ap

In [ ]:
df_ap = df_ap.drop(columns=['Age','Sex','bmi_lt_25']).join(pd.get_dummies(df_ap.loc[:,['Age','Sex']], drop_first=True))
df_ap

In [ ]:
X_ap = df_ap.iloc[:,1:].to_numpy()
X_ap = np.concatenate([np.ones(shape=(X_ap.shape[0],1)),X_ap],axis=1)
y_ap = df_ap.iloc[:,0].to_numpy()

In [ ]:
if LOAD:
    w_true_ap = np.load('saved_data/w_true_ap.npy')
else:
    LR = LogisticRegression()
    LR.fit(X_ap[:,1:], y_ap)
    w_true_ap = np.ones(shape=(X_ap.shape[1],1))
    w_true_ap[0] = LR.intercept_
    w_true_ap[1:] = LR.coef_.reshape(-1, 1)

In [ ]:
logit_true_ap = X_ap @ w_true_ap
probs_true_ap = expit(logit_true_ap)
om_probs_true_ap = expit(-logit_true_ap)

### Initial logreg on semi-synthetic data

In [ ]:
if LOAD:
    logit_pred_ap = np.load('saved_data/logit_pred_ap.npy')
    y_obs_ap      = np.load('saved_data/y_obs_ap.npy')

else:
    logit_pred_ap = np.zeros((X_ap.shape[0], NUM_TRIALS))
    y_obs_ap = np.zeros((X_ap.shape[0], NUM_TRIALS))
    
    for i in range(NUM_TRIALS):
        LR = LogisticRegression(C=np.inf, fit_intercept=False, solver='lbfgs')
        y_obs_ap[:,i] = rng.binomial(1, probs_true_ap[:,0])
        LR.fit(X_ap, y_obs_ap[:,i])
        w = LR.coef_.reshape(-1, 1)
    
        logit_pred_ap[:,i] = (X_ap @ w).reshape(-1)

In [ ]:
probs_pred_ap = expit(logit_pred_ap)
om_probs_pred_ap = expit(-logit_pred_ap)

### Compute regret

In [ ]:
if LOAD:
    resamp_from_true_logodds_ap = np.load('saved_data/resamp_from_true_logodds_ap.npy')
    resamp_from_pred_logodds_ap = np.load('saved_data/resamp_from_pred_logodds_ap.npy')

else:
    resamp_from_true_logodds_ap = np.zeros((X_ap.shape[0], NUM_TRIALS, NUM_RESAMPLES))
    resamp_from_pred_logodds_ap = np.zeros((X_ap.shape[0], NUM_TRIALS, NUM_RESAMPLES))
    
    for i in tqdm(range(NUM_TRIALS)):
        
        for j in range(NUM_RESAMPLES):
            
            LR_from_true = LogisticRegression(C=np.inf, fit_intercept=False, solver='lbfgs')
            LR_from_true.fit(X_ap, rng.binomial(1, probs_true_ap[:,0]))
            w_from_true = LR_from_true.coef_.reshape(-1, 1)
                
            LR_from_pred = LogisticRegression(C=np.inf, fit_intercept=False, solver='lbfgs')
            LR_from_pred.fit(X_ap, rng.binomial(1, probs_pred_ap[:,i]))
            w_from_pred = LR_from_pred.coef_.reshape(-1, 1)
    
            resamp_from_true_logodds_ap[:,i,j] = (X_ap @ w_from_true).reshape(-1)
            resamp_from_pred_logodds_ap[:,i,j] = (X_ap @ w_from_pred).reshape(-1)

In [ ]:
regret_logit_true_ap = np.var(resamp_from_true_logodds_ap, axis=2, ddof=1)
regret_logit_pred_ap = np.var(resamp_from_pred_logodds_ap, axis=2, ddof=1)

regret_probs_true_ap = np.var(expit(resamp_from_true_logodds_ap), axis=2, ddof=1)
regret_probs_pred_ap = np.var(expit(resamp_from_pred_logodds_ap), axis=2, ddof=1)

### Save data

In [ ]:
if SAVE:
    np.save('saved_data/w_true_bc.npy', w_true_bc)
    np.save('saved_data/logit_pred_bc.npy', logit_pred_bc)
    np.save('saved_data/y_obs_bc.npy', y_obs_bc)
    np.save('saved_data/resamp_from_true_logodds_bc.npy', resamp_from_true_logodds_bc)
    np.save('saved_data/resamp_from_pred_logodds_bc.npy', resamp_from_pred_logodds_bc)

    np.save('saved_data/w_true_bk.npy', w_true_bk)
    np.save('saved_data/logit_pred_bk.npy', logit_pred_bk)
    np.save('saved_data/y_obs_bk.npy', y_obs_bk)
    np.save('saved_data/resamp_from_true_logodds_bk.npy', resamp_from_true_logodds_bk)
    np.save('saved_data/resamp_from_pred_logodds_bk.npy', resamp_from_pred_logodds_bk)

    np.save('saved_data/w_true_ap.npy', w_true_ap)
    np.save('saved_data/logit_pred_ap.npy', logit_pred_ap)
    np.save('saved_data/y_obs_ap.npy', y_obs_ap)
    np.save('saved_data/resamp_from_true_logodds_ap.npy', resamp_from_true_logodds_ap)
    np.save('saved_data/resamp_from_pred_logodds_ap.npy', resamp_from_pred_logodds_ap)

In [ ]:
Hessian_bc = \
   (X_bc.reshape((1,) + X_bc.shape) * probs_true_bc.T.reshape(probs_true_bc.T.shape + (1,))).transpose([0,2,1]) \
  @ (om_probs_true_bc.T.reshape(om_probs_true_bc.T.shape + (1,)) * X_bc.reshape((1,) + X_bc.shape))

regret_logit_theory_bc =  \
   np.sum(X_bc.T.reshape((1,) + X_bc.T.shape) * np.linalg.solve(Hessian_bc, X_bc.T.reshape((1,) + X_bc.T.shape)), axis=1).T
regret_probs_theory_bc = probs_true_bc**2 * om_probs_true_bc**2 * regret_logit_theory_bc

Hessian_bk = \
   (X_bk.reshape((1,) + X_bk.shape) * probs_true_bk.T.reshape(probs_true_bk.T.shape + (1,))).transpose([0,2,1]) \
  @ (om_probs_true_bk.T.reshape(om_probs_true_bk.T.shape + (1,)) * X_bk.reshape((1,) + X_bk.shape))

Hessian_bk

regret_logit_theory_bk =  \
   np.sum(X_bk.T.reshape((1,) + X_bk.T.shape) * np.linalg.solve(Hessian_bk, X_bk.T.reshape((1,) + X_bk.T.shape)), axis=1).T

regret_probs_theory_bk = probs_true_bk**2 * om_probs_true_bk**2 * regret_logit_theory_bk

Hessian_ap = \
   (X_ap.reshape((1,) + X_ap.shape) * probs_true_ap.T.reshape(probs_true_ap.T.shape + (1,))).transpose([0,2,1]) \
  @ (om_probs_true_ap.T.reshape(om_probs_true_ap.T.shape + (1,)) * X_ap.reshape((1,) + X_ap.shape))

regret_logit_theory_ap =  \
   np.sum(X_ap.T.reshape((1,) + X_ap.T.shape) * np.linalg.solve(Hessian_ap, X_ap.T.reshape((1,) + X_ap.T.shape)), axis=1).T
regret_probs_theory_ap = probs_true_ap**2 * om_probs_true_ap**2 * regret_logit_theory_ap

In [ ]:
# subsample for plotting purposes, these points don't appear on the plot anyway
keep_inds_bk = np.abs(regret_probs_true_bk[:,0] - regret_probs_theory_bk[:,0]) > 0.00003
print(keep_inds_bk.sum(), keep_inds_bk.shape[0])

# Plots

### Single trial

In [ ]:
plt.rcParams.update({'font.size': FONT_SIZE})

plt.errorbar(regret_probs_true_bc[:,0],
             np.median(regret_probs_pred_bc, axis=1),
             yerr=np.abs(np.quantile(regret_probs_pred_bc, [0,1],axis=1) 
                        -np.median(regret_probs_pred_bc, axis=1).reshape(1,-1)),
             fmt='b.', ecolor=(0.9,0.9,1), zorder=0)
plt.errorbar(regret_probs_true_bc[:,0],
             np.median(regret_probs_pred_bc, axis=1),
             yerr=np.abs(np.quantile(regret_probs_pred_bc, [0.25,0.75],axis=1) 
                        -np.median(regret_probs_pred_bc, axis=1).reshape(1,-1)),
             fmt='b.', ecolor=(0.5,.5,1), zorder=1)
plt.ylabel("Regret (estimated)")
plt.xlabel("Regret (true)")

xlim = plt.gca().get_xlim()
ylim = plt.gca().get_ylim()
plt.plot([0,1],[0,1],'k--',zorder=2)
plt.xlim(xlim)
plt.ylim(ylim)

plt.gca().set_box_aspect(3/4)
plt.savefig('plots/regret-bc.pdf', bbox_inches='tight')
plt.show()

plt.errorbar(regret_probs_true_bk[keep_inds_bk,0],
             np.median(regret_probs_pred_bk[keep_inds_bk], axis=1),
             yerr=np.abs(np.quantile(regret_probs_pred_bk[keep_inds_bk], [0,1],axis=1) 
                        -np.median(regret_probs_pred_bk[keep_inds_bk], axis=1).reshape(1,-1)),
             fmt='b.', ecolor=(0.9,0.9,1), zorder=0)
plt.errorbar(regret_probs_true_bk[keep_inds_bk,0],
             np.median(regret_probs_pred_bk[keep_inds_bk], axis=1),
             yerr=np.abs(np.quantile(regret_probs_pred_bk[keep_inds_bk], [0.25,0.75],axis=1) 
                        -np.median(regret_probs_pred_bk[keep_inds_bk], axis=1).reshape(1,-1)),
             fmt='b.', ecolor=(0.5,0.5,1), zorder=0)
plt.ylabel("Regret (estimated)")
plt.xlabel("Regret (true)")

xlim = plt.gca().get_xlim()
ylim = plt.gca().get_ylim()
plt.plot([0,1],[0,1],'k--',zorder=1)
plt.xlim(xlim)
plt.ylim(ylim)

plt.gca().set_box_aspect(3/4)
plt.savefig('plots/regret-bk.pdf', bbox_inches='tight')
plt.show()

plt.errorbar(regret_probs_true_ap[:,0],
             np.median(regret_probs_pred_ap, axis=1),
             yerr=np.abs(np.quantile(regret_probs_pred_ap, [0,1],axis=1) 
                        -np.median(regret_probs_pred_ap, axis=1).reshape(1,-1)),
             fmt='b.', ecolor=(0.9,0.9,1), zorder=0)
plt.errorbar(regret_probs_true_ap[:,0],
             np.median(regret_probs_pred_ap, axis=1),
             yerr=np.abs(np.quantile(regret_probs_pred_ap, [0.25,0.75],axis=1) 
                        -np.median(regret_probs_pred_ap, axis=1).reshape(1,-1)),
             fmt='b.', ecolor=(0.5,0.5,1), zorder=0)
plt.ylabel("Regret (estimated)")
plt.xlabel("Regret (true)")

xlim = plt.gca().get_xlim()
ylim = plt.gca().get_ylim()
plt.plot([0,1],[0,1],'k--',zorder=1)
plt.xlim(xlim)
plt.ylim(ylim)

plt.gca().set_box_aspect(3/4)
plt.savefig('plots/regret-ap.pdf', bbox_inches='tight')
plt.show()

# Theory

In [ ]:
plt.rcParams.update({'font.size': FONT_SIZE})
plt.scatter(regret_probs_true_bc[:,0],
            np.median(regret_probs_theory_bc, axis=1),
            color='b', marker='.', zorder=0)
plt.ylabel("Regret (theory)")
plt.xlabel("Regret (true)")

xlim = plt.gca().get_xlim()
ylim = plt.gca().get_ylim()
plt.plot([0,1],[0,1],'k--',zorder=1)
plt.xlim(xlim)
plt.ylim(ylim)

plt.gca().set_box_aspect(3/4)
plt.savefig('plots/regret-theory-bc.pdf', bbox_inches='tight')
plt.show()

plt.rcParams.update({'font.size': FONT_SIZE})
plt.scatter(regret_probs_true_bk[keep_inds_bk,0],
            np.median(regret_probs_theory_bk[keep_inds_bk], axis=1),
            color='b', marker='.', zorder=0)
plt.ylabel("Regret (theory)")
plt.xlabel("Regret (true)")

xlim = plt.gca().get_xlim()
ylim = plt.gca().get_ylim()
plt.plot([0,1],[0,1],'k--',zorder=1)
plt.xlim(xlim)
plt.ylim(ylim)

plt.gca().set_box_aspect(3/4)
plt.savefig('plots/regret-theory-bk.pdf', bbox_inches='tight')
plt.show()

plt.rcParams.update({'font.size': FONT_SIZE})
plt.scatter(regret_probs_true_ap[:,0],
            np.median(regret_probs_theory_ap, axis=1),
            color='b', marker='.', zorder=0)
plt.ylabel("Regret (theory)")
plt.xlabel("Regret (true)")

xlim = plt.gca().get_xlim()
ylim = plt.gca().get_ylim()
plt.plot([0,1],[0,1],'k--',zorder=1)
plt.xlim(xlim)
plt.ylim(ylim)

plt.gca().set_box_aspect(3/4)
plt.savefig('plots/regret-theory-ap.pdf', bbox_inches='tight')
plt.show()

# Other

In [ ]:
groups = [keep_inds_bk,
          df_bk.loc[:,'euribor3m_geq_500']==1,
          df_bk.loc[:,'euribor3m_geq_300']-df_bk.loc[:,'euribor3m_geq_400']==1,
          df_bk.loc[:,'married'] + df_bk.loc[:,'single'] + df_bk.loc[:,'divorced'] == 0]
colors = ['b','r','g', 'k']
markers = ['.', '^', 'x', 'v']

plt.rcParams.update({'font.size': FONT_SIZE})
for g, c, m in zip(groups, colors, markers):
    plt.scatter(regret_probs_true_bk[g,0],
                np.median(regret_probs_theory_bk[g], axis=1),
                color=c, marker=m, zorder=0)
plt.ylabel("Regret (theory)")
plt.xlabel("Regret (true)")

xlim = plt.gca().get_xlim()
ylim = plt.gca().get_ylim()
plt.plot([0,1],[0,1],'k--',zorder=1)
plt.xlim(xlim)
plt.ylim(ylim)

plt.legend(['Group 0', 'Group 1', 'Group 2', 'Group 3'])

plt.savefig('plots/regret-theory-bk-GROUP.pdf', bbox_inches='tight')

plt.gca().set_box_aspect(3/4)
plt.show()

In [ ]:
[g.sum() for g in groups[1:]]